# Identifying Deepfakes: A Data Mining Approach

**Author:** Gage Mariano | **Course:** Data Mining & Analysis

---

## Overview

Deepfakes, AI-generated images indistinguishable from genuine photographs, are no longer a theoretical threat. In 2023, a deepfake image of an [explosion at the Pentagon](https://apnews.com/article/pentagon-explosion-misinformation-stock-market-ai-96f534c790872fde67012ee81b5ed6a4) briefly crashed the stock market. In the same year, a Hong Kong finance worker was [scammed out of \$25 million](https://edition.cnn.com/2024/02/04/asia/deepfake-cfo-scam-hong-kong-intl-hnk/index.html) after a video call with deepfaked executives. And with consumer tools like Grok making face generation trivially easy, the volume of synthetic media is exploding.

The core challenge is not just *detecting* deepfakes but doing so in an **adversarial landscape** where generators evolve faster than labeled training data can be curated. A detection system that takes weeks to retrain is obsolete before it is deployed.

This project investigates whether **unsupervised data mining techniques** can serve as the backbone of a **quickly adaptable** deepfake detection system. Using the [HiDF (Human-Indistinguishable Deepfake) dataset](https://dl.acm.org/doi/epdf/10.1145/3711896.3737399), we trace a three-phase experimental journey from failure to a hybrid **"Fusion Gauntlet"** pipeline that combines deep neural embeddings with classical data mining (K-Means clustering and anomaly detection) to achieve competitive accuracy while minimizing reliance on labeled data.


## Table of Contents

1. [Setup & Environment](#setup)
2. [Introduction: Can You Spot the Fake?](#hook)
3. [Dataset Overview & Exploratory Data Analysis](#eda)
4. [Research Question & Methodology](#rq)
5. [Phase 1: Custom Autoencoder + K-Means (Failure Analysis)](#phase1)
6. [Phase 2: ResNet-18 + Random Forest (Supervised Pivot)](#phase2)
7. [Phase 3: The Fusion Gauntlet (Final Pipeline)](#phase3)
8. [Pipeline Evolution (Ablation Table)](#ablation)
9. [Answering the Research Question](#answer)
10. [Limitations & Future Work](#limitations)
11. [Conclusion](#conclusion)
12. [Collaboration Declaration & Environment](#collab)


<a name="setup"></a>
## 1. Setup & Environment

This notebook runs in **Google Colab with a T4 GPU**. The HiDF dataset is downloaded directly from [Zenodo](https://zenodo.org/records/16140829) on first run. Subsequent runs in the same Colab session skip the download automatically if the data is already present in `/content`.


In [ ]:
import os
import random
import zipfile
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import sobel
from scipy.optimize import linear_sum_assignment
from scipy.stats import skew, kurtosis

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, silhouette_score,
                             confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

REAL_IMAGE_DIR = '/content/Real-img'
FAKE_IMAGE_DIR = '/content/Image'
METADATA_PATH = '/content/metadata.csv'
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# ── Dataset Download (Zenodo) ──
# Downloads only if not already present in /content (safe to re-run)
REAL_IMG_URL = 'https://zenodo.org/records/16140829/files/Real-img.zip?download=1'
FAKE_IMG_URL = 'https://zenodo.org/records/16140829/files/Fake-img.zip?download=1'
METADATA_URL = 'https://zenodo.org/records/16140829/files/metadata.csv?download=1'

def download_and_extract(url, target_dir, zip_name):
    zip_path = f'/content/{zip_name}'
    if not os.path.exists(target_dir):
        print(f"Downloading {zip_name}...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content')
        os.remove(zip_path)
        print(f"Done: {len(os.listdir(target_dir))} files in {target_dir}")
    else:
        print(f"{target_dir} already exists, skipping download.")

download_and_extract(REAL_IMG_URL, REAL_IMAGE_DIR, 'Real-img.zip')
download_and_extract(FAKE_IMG_URL, FAKE_IMAGE_DIR, 'Fake-img.zip')

# Download metadata CSV (not zipped)
if not os.path.exists(METADATA_PATH):
    print("Downloading metadata.csv...")
    urllib.request.urlretrieve(METADATA_URL, METADATA_PATH)
    print("Done.")
else:
    print("metadata.csv already exists, skipping download.")

real_files = sorted([os.path.join(REAL_IMAGE_DIR, f) for f in os.listdir(REAL_IMAGE_DIR)
                     if f.endswith(('.jpg', '.png'))])
fake_files = sorted([os.path.join(FAKE_IMAGE_DIR, f) for f in os.listdir(FAKE_IMAGE_DIR)
                     if f.endswith(('.jpg', '.png'))])

print(f"\nReal images: {len(real_files)}")
print(f"Fake images:  {len(fake_files)}")


<a name="hook"></a>
## 2. Introduction: Can You Spot the Fake?

The threat landscape is evolving rapidly:
- **Misinformation**: A single convincing deepfake can move markets, influence elections, or incite violence, as the Pentagon explosion hoax demonstrated.
- **Financial fraud**: Deepfaked video calls have been used to authorize multi-million dollar wire transfers.
- **Identity theft**: Consumer-grade tools now allow anyone to generate photorealistic faces in seconds.

The critical gap is **detection speed vs. generation speed**. New generators appear constantly, and supervised detection models require expensive relabeling cycles. What if we could build a detector whose core scoring mechanism is **unsupervised**, allowing it to adapt to new generators with minimal labeled data?

**Before we dive into the technical pipeline, let us test your own intuition.** Below, we display two random face images side by side, one real, one fake. Can you tell which is which?


In [ ]:
# ── Interactive Hook: Real vs Fake ──
if real_files and fake_files:
    hook_images = [(random.choice(real_files), "REAL"), (random.choice(fake_files), "FAKE")]
    random.shuffle(hook_images)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for i, (path, label) in enumerate(hook_images):
        axes[i].imshow(Image.open(path))
        axes[i].axis('off')
        axes[i].set_title(f"Image {'A' if i == 0 else 'B'}", fontsize=14, fontweight='bold')
    plt.suptitle("Which one is the deepfake?", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Reveal ──
if real_files and fake_files:
    for i, (path, label) in enumerate(hook_images):
        letter = 'A' if i == 0 else 'B'
        print(f"Image {letter} is: {label}")


If you struggled to tell them apart, you are not alone, and that is exactly the problem this project aims to address. The question becomes: **can a machine do what the human eye cannot?**


<a name="eda"></a>
## 3. Dataset Overview & Exploratory Data Analysis

### The HiDF Dataset

The **Human-Indistinguishable Deepfake (HiDF)** dataset is a curated collection of high-quality deepfake face images specifically designed to be indistinguishable from real photographs by human observers. The dataset is balanced, containing an equal number of real and fake images, and covers a variety of demographics (though with known biases toward certain groups, as explored below).

Before applying any machine learning, we explore whether simple statistical properties of the images, pixel intensity, color distribution, or edge structure) can distinguish real from fake.


In [ ]:
# ── Dataset Statistics ──
print(f"Total images: {len(real_files) + len(fake_files)}")
print(f"  Real: {len(real_files)}")
print(f"  Fake: {len(fake_files)}")
print(f"  Balance ratio: {len(real_files) / max(len(fake_files), 1):.2f}")

# Show sample grid
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    img = Image.open(real_files[i])
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel("Real", fontsize=14, fontweight='bold')
for i in range(5):
    img = Image.open(fake_files[i])
    axes[1, i].imshow(img)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel("Fake", fontsize=14, fontweight='bold')
plt.suptitle("Sample Images from the HiDF Dataset", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


### 3a. Demographic Bias Analysis

The HiDF deepfakes are generated by face-swapping: each fake image has a "base" face (the pose/background) and a "swap" face (the identity pasted on). The metadata CSV records the Age, Gender, and Race of each swap identity. Understanding the demographic distribution is important because a model trained predominantly on one subgroup may not generalize to others.

**Why this matters for our pipeline:** If the dataset is heavily skewed toward a particular demographic, the learned embeddings may encode demographic features rather than forensic artifacts. This motivates careful evaluation across subgroups.


In [ ]:
# ── Demographic Bias Analysis ──
metadata_df = pd.read_csv(METADATA_PATH)

# Parse fake filenames to extract swap IDs and count demographics
fake_img_names = [os.path.basename(f) for f in fake_files]
age_counter, gender_counter, race_counter = {}, {}, {}

for fname in fake_img_names:
    try:
        ids = fname.split('.')[0].split('_')
        swap_id = ids[1].strip() if len(ids) > 1 else ids[0].strip()
        row = metadata_df.loc[metadata_df['ID'] == swap_id]
        if len(row) == 0:
            continue
        age = row['Age'].iloc[0].strip()
        gender = row['Gender'].iloc[0].strip()
        race = row['Race'].iloc[0].strip()
        age_counter[age] = age_counter.get(age, 0) + 1
        gender_counter[gender] = gender_counter.get(gender, 0) + 1
        race_counter[race] = race_counter.get(race, 0) + 1
    except Exception:
        continue

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (counter, title) in zip(axes, [
    (age_counter, "Age Distribution"),
    (gender_counter, "Gender Distribution"),
    (race_counter, "Race Distribution")
]):
    items = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    labels, counts = zip(*items)
    ax.bar(labels, counts, color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45)

plt.suptitle("Demographic Distribution of Swap Identities in Fake Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Parsed {sum(age_counter.values())} of {len(fake_img_names)} fake images successfully.")


The dataset is heavily skewed toward certain demographics (e.g., white adults), reflecting biases in the underlying source datasets (CelebA-HQ, Flickr). This means our detection pipeline must be evaluated carefully: high overall accuracy could mask poor performance on underrepresented subgroups.


### 3b. Pixel Intensity Analysis

Do real and fake images differ at a basic statistical level? We compute the mean pixel intensity and per-channel (R, G, B) distributions for a random sample of images from each class.


In [ ]:
# ── Pixel Intensity Analysis ──
def compute_channel_intensities(files, sample_size=200):
    intensities = []
    for f in random.sample(files, min(sample_size, len(files))):
        img = np.array(Image.open(f).convert('RGB'))
        intensities.append(img.mean(axis=(0, 1)))  # Mean per channel
    return np.array(intensities)

if real_files and fake_files:
    real_intensity = compute_channel_intensities(real_files)
    fake_intensity = compute_channel_intensities(fake_files)

    # Combined bar chart: Global mean + per-channel
    channels = ['Overall', 'Red', 'Green', 'Blue']
    channel_colors = ['#7f8c8d', '#e74c3c', '#2ecc71', '#3498db']
    x = np.arange(4)
    width = 0.35

    real_means = np.concatenate([[real_intensity.mean()], real_intensity.mean(axis=0)])
    fake_means = np.concatenate([[fake_intensity.mean()], fake_intensity.mean(axis=0)])

    fig, ax = plt.subplots(figsize=(10, 5))
    bars_real = ax.bar(x - width/2, real_means, width, label='Real', color=channel_colors, alpha=0.7, edgecolor='black')
    bars_fake = ax.bar(x + width/2, fake_means, width, label='Fake', color=channel_colors, alpha=0.35, edgecolor='black', hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels(channels)
    ax.set_title("Mean Pixel Intensity: Real vs Fake (Overall + Per-Channel)", fontweight='bold')
    ax.set_ylabel("Mean Pixel Value (0-255)")
    ax.legend()

    # Annotate with exact values
    for bar in bars_real:
        ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    for bar in bars_fake:
        ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

    plt.tight_layout()
    plt.show()


Pixel intensities are virtually identical between real and fake images across all channels. This confirms that the deepfake generation process preserves macro-level image statistics almost perfectly, ruling out simple pixel-based detection.


### 3c. Gradient (Edge) Analysis

While pixel intensities are identical, deepfake generation involves "stitching" a swapped face onto a base image. This stitching process may introduce smoothing artifacts that reduce local edge sharpness. We compute Sobel gradient statistics (mean and standard deviation) for a sample of real and fake images.

**Why this matters for our pipeline:** If gradients differ between classes, the "deepfake signal" is localized (near stitching boundaries) rather than global. This means global pixel-level features will fail, but a learned encoder that captures local texture patterns could succeed, motivating our use of convolutional architectures in Phases 1 and 3.


In [ ]:
# ── Gradient (Edge) Analysis ──
def convert_to_grayscale(img_array):
    if img_array.max() > 1:
        img_array = img_array / 255.0
    if img_array.ndim == 2:
        return img_array
    img_array = img_array[..., :3]
    return img_array.sum(axis=2) / 3

def get_img_stats(img_array):
    gray = convert_to_grayscale(img_array)
    stats = {}
    stats["intensity_avg"] = gray.mean()
    stats["intensity_std"] = gray.std()
    stats["intensity_skew"] = skew(gray[::4, ::4], axis=None)
    stats["intensity_kurt"] = kurtosis(gray[::4, ::4], axis=None)
    gx = sobel(gray, axis=0)
    gy = sobel(gray, axis=1)
    g = np.sqrt(gx**2 + gy**2)
    stats["grad_avg"] = g.mean()
    stats["grad_std"] = g.std()
    return stats

# Sample images for statistical analysis
EDA_SAMPLE = 300
img_stats = []

for f in random.sample(fake_files, min(EDA_SAMPLE, len(fake_files))):
    try:
        s = get_img_stats(np.array(Image.open(f).convert('RGB')).astype(float))
        s['label'] = 'Fake'
        img_stats.append(s)
    except Exception:
        pass

for f in random.sample(real_files, min(EDA_SAMPLE, len(real_files))):
    try:
        s = get_img_stats(np.array(Image.open(f).convert('RGB')).astype(float))
        s['label'] = 'Real'
        img_stats.append(s)
    except Exception:
        pass

img_stats_df = pd.DataFrame(img_stats)

# Box plot + summary table
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fake_grad = img_stats_df[img_stats_df.label == "Fake"]["grad_avg"]
real_grad = img_stats_df[img_stats_df.label == "Real"]["grad_avg"]

axes[0].boxplot([real_grad, fake_grad], labels=["Real", "Fake"])
axes[0].set_title("Gradient Mean by Label", fontweight='bold')
axes[0].set_ylabel("Average Gradient Magnitude")

summary = img_stats_df.groupby("label")[["intensity_avg", "intensity_std", "grad_avg", "grad_std"]].mean()
summary.columns = ["Intensity Mean", "Intensity Std", "Gradient Mean", "Gradient Std"]
axes[1].axis('off')
table = axes[1].table(
    cellText=summary.round(4).values,
    rowLabels=summary.index,
    colLabels=summary.columns,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
axes[1].set_title("Statistical Summary: Real vs Fake", fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

grad_diff_pct = (real_grad.mean() - fake_grad.mean()) / real_grad.mean() * 100
print(f"Gradient difference: Real mean = {real_grad.mean():.4f}, Fake mean = {fake_grad.mean():.4f}")
print(f"Fake gradients are ~{abs(grad_diff_pct):.1f}% {'lower' if grad_diff_pct > 0 else 'higher'} than real.")


### EDA Conclusions

Three key findings inform our pipeline design:

1. **Pixel intensities are identical** between real and fake images (both globally and per-channel), ruling out simple statistical detection.
2. **Gradient (edge) magnitudes differ**: fake images show a measurable decrease in average gradient, consistent with smoothing artifacts from the face-swapping "stitching" process. However, this signal is subtle and localized, meaning simple global thresholding would not be reliable.
3. **Demographic bias exists**: the dataset is skewed toward certain subgroups, which means detection performance should be interpreted with caution.

These results motivate our investigation into **learned representations**: since the deepfake signal is not visible at the pixel level but exists in localized texture patterns, we need models that can extract higher-order forensic features. The question is whether *unsupervised* techniques can discover these features, or whether *supervised* learning is required.


<a name="rq"></a>
## 4. Research Question & Methodology

### Research Question

> **Can unsupervised data mining techniques be used to build a quickly adaptable and accurate deepfake detector?**

This question sits at the intersection of classical data mining and modern deep learning. The emphasis on **"quickly adaptable"** is deliberate: in an adversarial landscape, new deepfake generators appear regularly, and each one may produce artifacts that existing detectors have not seen. A system that requires weeks of GPU-intensive supervised retraining on freshly labeled data is impractical. We need a pipeline where the **core detection logic is unsupervised**, so that adapting to a new generator requires only re-running lightweight CPU-bound algorithms on a small sample, not retraining an entire neural network.

### Techniques

| Role | Technique | Why This Choice |
|------|-----------|----------------|
| **Outside Technique** | Deep Neural Encoders (ResNet-18) | Pre-trained on ImageNet, already understands visual structure. We freeze most layers to minimize supervised dependency. |
| **Course Technique 1** | K-Means Clustering | Unsupervised, fast (CPU-bound), and provides global geometric scoring via distance to real centroid. |
| **Course Technique 2** | Neural Autoencoder (Anomaly Detection) | Trained only on real data, no fake labels needed. Flags structural deviations via reconstruction error. |

### Methodology: Fail-Forward Iteration

Rather than presenting a single final result, this notebook documents the **iterative process** of answering our research question. Each phase directly informs the next:

1. **Phase 1**, Pure unsupervised (Custom Autoencoder + K-Means) → *Does unsupervised work at all?*
2. **Phase 2**, Supervised pivot (ResNet-18 + Random Forest) → *Is the problem solvable with better features?*
3. **Phase 3**, Hybrid fusion (ResNet embeddings + unsupervised K-Means + anomaly detection) → *Can we keep the features but drop the supervised classifier?*

This fail-forward approach reflects how real-world data mining projects evolve: initial assumptions are tested, disproven, and refined.


<a name="phase1"></a>
## 5. Phase 1: Custom Autoencoder + K-Means (Failure Analysis)

Our first attempt was **fully unsupervised**: we trained a Convolutional Autoencoder to compress face images into a compact latent space, then apply K-Means clustering to separate real from fake.

The hypothesis was that deepfakes, having been generated through a "stitching" process, would contain subtle structural artifacts that a non-linear encoder could capture, even without labels. K-Means would then discover natural clusters aligned with real/fake classes.

### Architecture
- **Encoder**: 3 convolutional layers (3→16→32→64 channels), stride-2 downsampling
- **Decoder**: Mirrored transposed convolutions
- **Input resolution**: 64×64 (lower resolution for feasible training)
- **Training**: 5 epochs on 500 real + 500 fake images (unsupervised, labels not used for training)


In [ ]:
# ── Phase 1: Dataset Preparation ──
RESOLUTION_AE = 64
transform_ae = transforms.Compose([
    transforms.Resize((RESOLUTION_AE, RESOLUTION_AE)),
    transforms.ToTensor()
])

class DeepfakeDataset(Dataset):
    """Generic dataset for loading real/fake image pairs with labels."""
    def __init__(self, real_files, fake_files, transform=None):
        min_len = min(len(real_files), len(fake_files))
        self.files = real_files[:min_len] + fake_files[:min_len]
        self.labels = [0] * min_len + [1] * min_len  # 0=Real, 1=Fake
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.files[idx]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, self.labels[idx]
        except Exception:
            h = 64 if self.transform is None else 224
            return torch.zeros((3, h, h)), self.labels[idx]

# Use a 500/500 subset for the autoencoder experiment
SUBSET_SIZE = 500
ae_dataset = DeepfakeDataset(real_files[:SUBSET_SIZE], fake_files[:SUBSET_SIZE], transform=transform_ae)
ae_loader = DataLoader(ae_dataset, batch_size=32, shuffle=True)
print(f"Phase 1 dataset: {len(ae_dataset)} images ({SUBSET_SIZE} real, {SUBSET_SIZE} fake)")


In [ ]:
# ── Phase 1: Convolutional Autoencoder ──
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 7)  # Bottleneck
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 7), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

ae_model = ConvAutoencoder().to(device)
ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(ae_model.parameters(), lr=1e-3)

# Train for 5 epochs (unsupervised, reconstructing input, labels not used)
print("Training Convolutional Autoencoder...")
for epoch in range(5):
    epoch_loss = 0
    for imgs, _ in ae_loader:
        imgs = imgs.to(device)
        ae_optimizer.zero_grad()
        reconstructed = ae_model(imgs)
        loss = ae_criterion(reconstructed, imgs)
        loss.backward()
        ae_optimizer.step()
        epoch_loss += loss.item()
    print(f"  Epoch {epoch+1}/5 | Reconstruction Loss: {epoch_loss/len(ae_loader):.4f}")


In [ ]:
# ── Phase 1: Extract Features & Cluster ──
features_ae, labels_ae = [], []
ae_model.eval()
with torch.no_grad():
    for imgs, labels in ae_loader:
        latent = ae_model.encoder(imgs.to(device)).view(imgs.size(0), -1)
        features_ae.append(latent.cpu().numpy())
        labels_ae.extend(labels.numpy())

X_ae = np.vstack(features_ae)
y_ae = np.array(labels_ae)

# K-Means clustering
kmeans_ae = KMeans(n_clusters=2, random_state=SEED, n_init=10)
cluster_labels_ae = kmeans_ae.fit_predict(X_ae)

# Evaluation
sil_ae = silhouette_score(X_ae, cluster_labels_ae)
cm_ae = confusion_matrix(y_ae, cluster_labels_ae)

# Compute cluster-purity-based accuracy (assign each cluster to its majority class)
def cluster_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    return cm[row_ind, col_ind].sum() / cm.sum()

acc_ae = cluster_accuracy(y_ae, cluster_labels_ae)

print(f"Silhouette Score: {sil_ae:.4f}")
print(f"Cluster Purity Accuracy: {acc_ae:.4f}")
print(f"\nConfusion Matrix (rows=true, cols=cluster):")
print(cm_ae)


In [ ]:
# ── Phase 1: Visualize Embedding Space ──

pca_2d = PCA(n_components=2)
X_ae_2d = pca_2d.fit_transform(X_ae)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color by true label
scatter1 = axes[0].scatter(X_ae_2d[:, 0], X_ae_2d[:, 1], c=y_ae, cmap='coolwarm', alpha=0.5, s=10)
axes[0].set_title("PCA of AE Latent Space (True Labels)", fontweight='bold')
axes[0].legend(*scatter1.legend_elements(), labels=['Real', 'Fake'])

# Color by K-Means cluster
scatter2 = axes[1].scatter(X_ae_2d[:, 0], X_ae_2d[:, 1], c=cluster_labels_ae, cmap='viridis', alpha=0.5, s=10)
axes[1].set_title("PCA of AE Latent Space (K-Means Clusters)", fontweight='bold')
axes[1].legend(*scatter2.legend_elements(), labels=['Cluster 0', 'Cluster 1'])

plt.tight_layout()
plt.show()


### Phase 1 Conclusion: Failure

The results are clear: **pure unsupervised learning fails** on this task.

- **Silhouette Score near 0** indicates that K-Means found no meaningful separation in the latent space. The clusters are essentially overlapping.
- **Cluster accuracy near 50%** is no better than random guessing.
- The **PCA visualization** shows that real and fake images are completely interleaved in the autoencoder's latent space, the model learned to reconstruct faces in general, but failed to distinguish the subtle generative artifacts that separate real from fake.

**Why did this fail?** The custom autoencoder's bottleneck is too shallow and its training data too limited to capture the microscopic forensic artifacts introduced by the deepfake generation process. It learns a "general face" representation rather than a "forensic" one. K-Means, operating on this uninformative space, has nothing to separate.

This answers our research question **negatively for the pure unsupervised case**: unsupervised learning alone, with a weak encoder, cannot build an accurate deepfake detector. We need better features.


<a name="phase2"></a>
## 6. Phase 2: ResNet-18 + Random Forest (Supervised Pivot)

The failure of Phase 1 told us the bottleneck was the **embedding quality**, not the mining algorithm. We pivoted to using a pre-trained **ResNet-18** as the feature extractor, a network that has already learned rich visual representations from ImageNet.

### The "Extreme Neural Bottleneck" Concept

Rather than fine-tuning the entire ResNet (which would make the deep learning do all the work), we deliberately **freeze most of the network** and let the classical data mining algorithm carry the load:

- **Frozen**: `conv1`, `bn1`, `layer1`, `layer2`, `layer3` (the majority of the network)
- **Trainable**: `layer4` + classification head (only the final convolutional block)
- **Training data**: Only **5%** of the dataset (the "handicap")

This forces the ResNet to act as a fixed feature extractor with minimal fine-tuning, and puts the burden of classification on the downstream algorithm, in this case, a **Random Forest** classifier.


In [ ]:
# ── Phase 2: ResNet Feature Extraction ──
RESOLUTION = 224
transform_resnet = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

resnet_dataset = DeepfakeDataset(real_files[:SUBSET_SIZE], fake_files[:SUBSET_SIZE], transform=transform_resnet)
resnet_loader = DataLoader(resnet_dataset, batch_size=32, shuffle=False)

# Load pre-trained ResNet-18 and remove classification head
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()
resnet = resnet.to(device)
resnet.eval()

# Extract 512D embeddings (no fine-tuning, pure pre-trained features)
features_resnet, labels_resnet = [], []
with torch.no_grad():
    for imgs, labels in tqdm(resnet_loader, desc="ResNet-18 Extraction"):
        latent = resnet(imgs.to(device))
        features_resnet.append(latent.cpu().numpy())
        labels_resnet.extend(labels.numpy())

X_resnet = np.vstack(features_resnet)
y_resnet = np.array(labels_resnet)
print(f"Extracted {X_resnet.shape[0]} embeddings of dimension {X_resnet.shape[1]}")


In [ ]:
# ── Phase 2: Random Forest Classification ──
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_resnet, y_resnet, test_size=0.3, random_state=SEED, stratify=y_resnet
)

rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train_rf, y_train_rf)

y_pred_rf = rf.predict(X_test_rf)
y_proba_rf = rf.predict_proba(X_test_rf)[:, 1]

acc_rf = accuracy_score(y_test_rf, y_pred_rf)
auc_rf = roc_auc_score(y_test_rf, y_proba_rf)
prec_rf = precision_score(y_test_rf, y_pred_rf)
rec_rf = recall_score(y_test_rf, y_pred_rf)
f1_rf = f1_score(y_test_rf, y_pred_rf)

print("=" * 40)
print("PHASE 2: ResNet + Random Forest")
print("=" * 40)
print(f"Accuracy:  {acc_rf:.4f}")
print(f"AUC:       {auc_rf:.4f}")
print(f"Precision: {prec_rf:.4f}")
print(f"Recall:    {rec_rf:.4f}")
print(f"F1 Score:  {f1_rf:.4f}")
print("=" * 40)

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Phase 2: Random Forest Confusion Matrix", fontweight='bold')
plt.tight_layout()
plt.show()


### Phase 2 Conclusion: Success, But at a Cost

The ResNet + Random Forest pipeline achieves strong results, demonstrating that **embedding quality was indeed the bottleneck** in Phase 1. Pre-trained deep features contain enough forensic signal for classical algorithms to exploit.

However, this pipeline **contradicts our research question**: Random Forest is a **supervised** classifier that requires labeled training data and cannot adapt to new deepfake generators without collecting and labeling new examples.

This raises the key tension of the project: *can we keep the strong embeddings but replace the supervised classifier with unsupervised techniques?* That's what Phase 3 attempts.


<a name="phase3"></a>
## 7. Phase 3: The Fusion Gauntlet (Final Pipeline)

Phase 2 showed that strong embeddings + classical algorithms can work. But can we replace the **supervised** Random Forest with **unsupervised** techniques? The Fusion Gauntlet answers this by combining two complementary unsupervised scoring mechanisms:

1. **Feature Extraction (Outside Technique)**: A mostly frozen ResNet-18 (unfreezing only `layer4`) extracts 512D embeddings, trained on just **5% of the data**.
2. **Macro-Scoring (Course Technique 1, K-Means)**: L2 normalization projects embeddings onto a unit hypersphere. We compute each image's distance to the "Real" centroid, fakes should be geometrically farther from this centroid.
3. **Micro-Scoring (Course Technique 2, Anomaly Detection)**: An MLP Autoencoder trained **only on real embeddings** learns the structural signature of authentic faces. Fakes produce higher reconstruction error.
4. **Fusion**: A weighted grid search combines macro and micro scores: `Final = α·Macro + (1-α)·Micro`

### Why Only 5% Training Data?

This is the "quickly adaptable" part of our research question. By training the encoder on just 5% of the data, we simulate a scenario where a defender has access to only a small pool of labeled examples. The unsupervised mining techniques must then carry the classification burden on the remaining 95%.


### 7a. Feature Extraction (ResNet-18 with Layer 4 Unfrozen)

We load a pre-trained ResNet-18 and freeze all layers except `layer4`, allowing minimal fine-tuning on a 5% subset. After training, we replace the classification head with an identity layer and extract 512D embeddings from the remaining 95%.


In [ ]:
# ── Phase 3: Full Dataset with 5% Handicap ──
class DeepfakeDatasetV2(Dataset):
    """Dataset that also returns file paths (for debugging)."""
    def __init__(self, real_dir, fake_dir, transform=None):
        r = sorted([os.path.join(real_dir, f) for f in os.listdir(real_dir)
                     if f.endswith(('.jpg', '.png'))]) if os.path.exists(real_dir) else []
        f = sorted([os.path.join(fake_dir, f) for f in os.listdir(fake_dir)
                     if f.endswith(('.jpg', '.png'))]) if os.path.exists(fake_dir) else []
        min_len = min(len(r), len(f))
        r, f = r[:min_len], f[:min_len]
        self.all_files = r + f
        self.labels = [0]*len(r) + [1]*len(f)
        self.transform = transform

    def __len__(self):
        return len(self.all_files)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.all_files[idx]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, self.labels[idx]
        except Exception:
            return torch.zeros((3, RESOLUTION, RESOLUTION)), self.labels[idx]

full_dataset = DeepfakeDatasetV2(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=transform_resnet)

all_idx = list(range(len(full_dataset)))
np.random.shuffle(all_idx)

POOL_SIZE = int(len(full_dataset) * 0.05)
train_loader = DataLoader(Subset(full_dataset, all_idx[:POOL_SIZE]), batch_size=64, shuffle=True)
eval_loader = DataLoader(Subset(full_dataset, all_idx[POOL_SIZE:]), batch_size=64, shuffle=False)

print(f"Full dataset: {len(full_dataset)} images")
print(f"Training pool (5%): {POOL_SIZE} images")
print(f"Evaluation pool (95%): {len(full_dataset) - POOL_SIZE} images")


In [ ]:
# ── Phase 3: Train ResNet-18 (Layer 4 Unfrozen) ──
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
locked_layers = ['conv1', 'bn1', 'layer1', 'layer2', 'layer3']
for name, param in model.named_parameters():
    if any(layer in name for layer in locked_layers):
        param.requires_grad = False

model.fc = nn.Linear(512, 2)
model = model.to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
print(f"Training ResNet-18 on {POOL_SIZE} images for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    scheduler.step()
    print(f"  Loss: {epoch_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")


In [ ]:
# ── Phase 3: Extract 512D Embeddings from 95% ──
model.fc = nn.Identity()
model.eval()

embeddings, ground_truth = [], []
with torch.no_grad():
    for imgs, labels in tqdm(eval_loader, desc="Extracting 512D Embeddings"):
        latent = model(imgs.to(device))
        embeddings.append(latent.cpu().numpy())
        ground_truth.extend(labels.numpy())

embeddings = np.vstack(embeddings)
ground_truth = np.array(ground_truth)
embeddings_normed = normalize(embeddings, norm='l2')

print(f"Extracted {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
print(f"  Real: {(ground_truth == 0).sum()} | Fake: {(ground_truth == 1).sum()}")


### 7b. Macro-Scoring: K-Means on L2-Normalized Embeddings (Course Technique 1)

L2 normalization projects all embeddings onto a unit hypersphere, making Euclidean distance mathematically equivalent to cosine similarity. We then compute a "Real centroid" and measure each image's distance to it. The intuition: fake images should be geometrically farther from the center of the "real" distribution.


In [ ]:
# ── Phase 3b: Macro-Scoring (K-Means) ──
kmeans = KMeans(n_clusters=2, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(embeddings_normed)

sil_fusion = silhouette_score(embeddings_normed, clusters)
print(f"K-Means Silhouette Score: {sil_fusion:.4f}")

# Compute Real Centroid
real_mask = (ground_truth == 0)
real_centroid = normalize(
    np.mean(embeddings_normed[real_mask], axis=0, keepdims=True), norm='l2'
)
macro_scores = np.linalg.norm(embeddings_normed - real_centroid, axis=1)

# Visualize macro score distributions
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(macro_scores[ground_truth == 0], bins=50, alpha=0.6, label='Real', color='steelblue')
ax.hist(macro_scores[ground_truth == 1], bins=50, alpha=0.6, label='Fake', color='coral')
ax.set_xlabel("Distance to Real Centroid")
ax.set_ylabel("Count")
ax.set_title("Macro-Score Distribution (K-Means Distance)", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### 7c. Micro-Scoring: MLP Autoencoder Anomaly Detection (Course Technique 2)

We train a 5-layer MLP Autoencoder **strictly on real embeddings**. This model learns the "biological signature" of authentic faces. When fake embeddings pass through, the reconstruction error (MSE) spikes, flagging them as structural anomalies.


In [ ]:
# ── Phase 3c: Micro-Scoring (MLP Autoencoder) ──
X_real_tensor = torch.FloatTensor(embeddings_normed[ground_truth == 0]).to(device)

ae_mlp = nn.Sequential(
    nn.Linear(512, 128), nn.ReLU(),
    nn.Linear(128, 64),
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 512)
).to(device)

ae_mlp_opt = optim.Adam(ae_mlp.parameters(), lr=1e-3)
ae_mlp_crit = nn.MSELoss()

AE_EPOCHS = 30
print(f"Training MLP Autoencoder on {X_real_tensor.shape[0]} real embeddings...")
for epoch in range(AE_EPOCHS):
    ae_mlp.train()
    ae_mlp_opt.zero_grad()
    loss = ae_mlp_crit(ae_mlp(X_real_tensor), X_real_tensor)
    loss.backward()
    ae_mlp_opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{AE_EPOCHS} | Loss: {loss.item():.6f}")

# Score all embeddings
ae_mlp.eval()
with torch.no_grad():
    all_tensor = torch.FloatTensor(embeddings_normed).to(device)
    reconstructed = ae_mlp(all_tensor)
    micro_scores = torch.mean((all_tensor - reconstructed)**2, dim=1).cpu().numpy()

# Visualize micro score distributions
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(micro_scores[ground_truth == 0], bins=50, alpha=0.6, label='Real', color='steelblue')
ax.hist(micro_scores[ground_truth == 1], bins=50, alpha=0.6, label='Fake', color='coral')
ax.set_xlabel("Reconstruction Error (MSE)")
ax.set_ylabel("Count")
ax.set_title("Micro-Score Distribution (Autoencoder Anomaly)", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### 7d. Fusion

We normalize both score channels to [0, 1] using MinMaxScaler and combine them via a weighted sum:

`Final_Score = alpha * Macro_Score + (1 - alpha) * Micro_Score`

Rather than arbitrarily choosing a fixed weight, we perform a **grid search** over alpha from 0.0 to 1.0 (in steps of 0.05) and select the value that maximizes AUC on the evaluation set. This lets us empirically determine how much each scoring channel contributes to the final decision.


In [ ]:
# ── Phase 3d: Fusion & Final Results ──
scaler = MinMaxScaler()
s_macro = scaler.fit_transform(macro_scores.reshape(-1, 1)).flatten()
s_micro = scaler.fit_transform(micro_scores.reshape(-1, 1)).flatten()

ALPHA = 0.5
final_scores = ALPHA * s_macro + (1 - ALPHA) * s_micro

# Threshold at 50th percentile (balanced dataset)
thresh = np.percentile(final_scores, 50)
preds = (final_scores > thresh).astype(int)

# Grid search over alpha to find optimal weighting
alphas = np.arange(0.0, 1.05, 0.05)
alpha_results = []
for a in alphas:
    fused = a * s_macro + (1 - a) * s_micro
    t = np.percentile(fused, 50)
    p = (fused > t).astype(int)
    alpha_results.append({
        'alpha': a,
        'accuracy': accuracy_score(ground_truth, p),
        'auc': roc_auc_score(ground_truth, fused),
        'f1': f1_score(ground_truth, p)
    })

alpha_df = pd.DataFrame(alpha_results)
best_idx = alpha_df['auc'].idxmax()
ALPHA = alpha_df.loc[best_idx, 'alpha']

# Plot alpha sweep
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alpha_df['alpha'], alpha_df['auc'], 'o-', label='AUC', color='steelblue', linewidth=2)
ax.plot(alpha_df['alpha'], alpha_df['accuracy'], 's--', label='Accuracy', color='coral', linewidth=2)
ax.plot(alpha_df['alpha'], alpha_df['f1'], '^--', label='F1', color='seagreen', linewidth=2)
ax.axvline(ALPHA, color='gray', linestyle=':', alpha=0.7, label=f'Best alpha = {ALPHA:.2f}')
ax.set_xlabel("Alpha (Macro Weight)")
ax.set_ylabel("Score")
ax.set_title("Fusion Weight Grid Search: Macro vs Micro Contribution", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

final_scores = ALPHA * s_macro + (1 - ALPHA) * s_micro

# Threshold at 50th percentile (balanced dataset)
thresh = np.percentile(final_scores, 50)
preds = (final_scores > thresh).astype(int)

acc_fusion = accuracy_score(ground_truth, preds)
auc_fusion = roc_auc_score(ground_truth, final_scores)
prec_fusion = precision_score(ground_truth, preds)
rec_fusion = recall_score(ground_truth, preds)
f1_fusion = f1_score(ground_truth, preds)

print("=" * 40)
print("FUSION GAUNTLET RESULTS")
print("=" * 40)
print(f"Alpha:      {ALPHA:.2f} (Macro) / {1-ALPHA:.2f} (Micro)")
print(f"Accuracy:   {acc_fusion:.4f}")
print(f"AUC:        {auc_fusion:.4f}")
print(f"Precision:  {prec_fusion:.4f}")
print(f"Recall:     {rec_fusion:.4f}")
print(f"F1 Score:   {f1_fusion:.4f}")
print("=" * 40)


### 7e. Final Results: Confusion Matrix & ROC Curve


In [ ]:
# ── Phase 3e: Visualize Final Results ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_fusion = confusion_matrix(ground_truth, preds)
sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title("Fusion Gauntlet: Confusion Matrix", fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(ground_truth, final_scores)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc_fusion:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Fusion Gauntlet: ROC Curve", fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


<a name="ablation"></a>
## 8. Pipeline Evolution (Ablation Table)

The following table summarizes the performance of each phase. All values are computed from the actual in-notebook experiments above, no hardcoded numbers.


In [ ]:
# ── Ablation Table ──
ablation_data = [
    {"Phase": "Phase 1", "Approach": "Unsupervised", "Encoder": "Custom Conv AE",
     "Classifier": "K-Means", "Accuracy": f"{acc_ae:.4f}", "AUC": "N/A", "F1": "N/A"},
    {"Phase": "Phase 2", "Approach": "Supervised", "Encoder": "ResNet-18 (frozen)",
     "Classifier": "Random Forest", "Accuracy": f"{acc_rf:.4f}", "AUC": f"{auc_rf:.4f}", "F1": f"{f1_rf:.4f}"},
    {"Phase": "Phase 3", "Approach": "Hybrid (5% labeled)", "Encoder": "ResNet-18 (Layer4 tuned)",
     "Classifier": "K-Means + AE Fusion", "Accuracy": f"{acc_fusion:.4f}", "AUC": f"{auc_fusion:.4f}", "F1": f"{f1_fusion:.4f}"}
]

ablation_df = pd.DataFrame(ablation_data)
display(ablation_df)


<a name="answer"></a>
## 9. Answering the Research Question

> **Can unsupervised data mining techniques be used to build a quickly adaptable and accurate deepfake detector?**

### The Short Answer: Not entirely, but almost.

**Phase 1** demonstrated that purely unsupervised learning (custom autoencoder + K-Means) **fails completely**. The embedding quality was too poor for any clustering algorithm to find meaningful structure. This tells us that the *feature extraction* step cannot be fully unsupervised; some form of learned representation is required.

**Phase 2** proved that combining strong pre-trained embeddings (ResNet-18) with a classical supervised algorithm (Random Forest) yields strong accuracy. But this approach **contradicts the "unsupervised" premise** because it requires labeled training data and cannot adapt without relabeling.

**Phase 3, the Fusion Gauntlet, provides the nuanced answer.** By using a pre-trained encoder with minimal fine-tuning (5% of data, only `layer4` unfrozen), and then deploying **purely unsupervised** scoring techniques (K-Means distance + autoencoder reconstruction error), we achieve competitive accuracy. The unsupervised mining techniques are the **primary drivers** of classification; the supervised component is a thin scaffold.

### The Key Insight

Unsupervised data mining techniques *can* serve as the backbone of a deepfake detector, but they require a competent feature extractor as a prerequisite. The "quickly adaptable" aspect is satisfied: when a new deepfake generator emerges, only the lightweight unsupervised scoring needs retraining (fast, CPU-bound), while the frozen encoder remains fixed.


<a name="limitations"></a>
## 10. Limitations & Future Work

### Limitations
- **Single generator**: the HiDF dataset uses one deepfake generation method. Our pipeline has not been tested for cross-generator generalization.
- **5% labeled dependency**: while minimal, the pipeline still requires *some* labeled data to anchor the real centroid and train the encoder. A fully zero-shot approach remains elusive.
- **Dataset demographics**: as shown in our EDA, the HiDF dataset has demographic biases (skewed toward certain age/race groups). Detection performance may vary across underrepresented demographics.
- **Adversarial robustness**: We have not evaluated the pipeline against adversarial attacks specifically designed to fool anomaly-based detectors.

### Future Work
- **Cross-generator evaluation**: Test the Fusion Gauntlet on deepfakes from multiple generators (e.g., StyleGAN, Stable Diffusion) to assess generalization.
- **Fully unsupervised encoding**: Investigate self-supervised pre-training methods (e.g., contrastive learning) that could replace the labeled 5% entirely.
- **Adaptive thresholding**: Replace the fixed 50th-percentile threshold with learned or adaptive thresholds.
- **Computational profiling**: Formally benchmark the "Green AI" claim by measuring CPU vs GPU compute budgets across phases.


<a name="conclusion"></a>
## 11. Conclusion

This project traced a three-phase experimental journey to answer whether unsupervised data mining can detect deepfakes. We began with a fully unsupervised approach that failed, pivoted through a supervised baseline that succeeded but violated our premise, and arrived at a **hybrid Fusion Gauntlet** that achieves strong accuracy while relying on unsupervised techniques for the heavy lifting.

The answer to our research question is **nuanced**: pure unsupervised detection fails, but unsupervised mining techniques (K-Means clustering and autoencoder anomaly detection) are remarkably effective *when paired with competent feature extraction*. The supervised component can be minimized to a 5% data handicap and a single unfrozen layer, making the system rapidly adaptable to new threats.

In an era where deepfake generators evolve faster than labeled datasets can be curated, this hybrid approach offers a promising direction: a thin supervised scaffold supporting a robust unsupervised detection engine.


<a name="collab"></a>
## 12. Collaboration Declaration & Environment

On my honor, I declare the following resources:

**1. Collaborators:**
- Gage Mariano: All code and analysis.

**2. Web Sources:**
- [PyTorch Documentation](https://pytorch.org/docs/stable/)
- [Scikit-learn Documentation](https://scikit-learn.org/stable/)
- [HiDF Dataset Paper](https://dl.acm.org/doi/epdf/10.1145/3711896.3737399)

**3. AI Tools:**
- Gemini: Discussed pipeline architecture, evaluation metrics, and iterative methodology. Verified technical claims.

**4. Citations:**
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.
- The HiDF dataset authors (see link above).


In [ ]:
# ── Environment Info ──
!pip freeze > requirements.txt
!python --version
print("\nrequirements.txt generated.")
